# exp_008 — QLoRA Fine-tune of Qwen3-4B

**Runtime:** Colab Pro — A100 40GB  
**Goal:** Fine-tune Qwen3-4B on NuminaMath-CoT + public correct responses, merge weights, push to HuggingFace.  
**Time estimate:** ~1.5–2 hrs for 2 epochs on 25K examples.

### Before running
1. Set `HF_TOKEN` and `HF_REPO_ID` in Cell 2.
2. Upload `public.jsonl`, `public_responses.jsonl`, and `results.jsonl` from exp_004 (see Cell 6 instructions).
3. Runtime → Change runtime type → A100 GPU.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl datasets bitsandbytes

In [ ]:
# Cell 2 — Config
# Add your HF token via Colab Secrets (left sidebar → key icon → add HF_TOKEN)
# This keeps it out of the notebook file entirely.
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

HF_REPO_ID   = "trevorduong/qwen3-4b-math-qlora"  # will be created as private

MODEL_NAME          = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH      = 2048
NUMINAMATH_N        = 25000   # examples to sample from NuminaMath-CoT
LORA_R              = 16
LORA_ALPHA          = 32
TRAIN_BATCH_SIZE    = 4
GRAD_ACCUM_STEPS    = 4       # effective batch = 16
NUM_EPOCHS          = 2
LEARNING_RATE       = 2e-4
RANDOM_SEED         = 42

In [ ]:
# Cell 3 — Load model in 4-bit + apply QLoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # auto-detect (bfloat16 on A100)
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
)
print(model.print_trainable_parameters())

In [ ]:
# Cell 4 — System prompts (exp_004 config — best found so far)
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

In [ ]:
# Cell 5 — Data formatting helpers
import re

def extract_boxed_and_reasoning(text):
    """Return (reasoning_text, boxed_expression) by splitting at the last \\boxed{."""
    idx = text.rfind(r'\boxed{')
    if idx == -1:
        return text.strip(), ""
    # Walk forward to find matching closing brace
    depth, end = 0, idx
    for i, ch in enumerate(text[idx:]):
        if ch == '{':  depth += 1
        elif ch == '}':  depth -= 1
        if depth == 0:
            end = idx + i + 1
            break
    boxed = text[idx:end]
    reasoning = text[:idx].strip()
    return reasoning, boxed


def make_messages(system, question, reasoning, boxed):
    """Build the ChatML messages list for one training example."""
    assistant = f"<think>\n{reasoning}\n</think>\n\n{boxed}"
    return [
        {"role": "system",    "content": system},
        {"role": "user",      "content": question},
        {"role": "assistant", "content": assistant},
    ]


def apply_chat_template(messages):
    """Render messages to a single string using the tokenizer's template."""
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


print("Helpers defined.")

In [ ]:
# Cell 6 — Load NuminaMath-CoT
from datasets import load_dataset
import random

random.seed(RANDOM_SEED)

print("Downloading NuminaMath-CoT (this may take a few minutes)...")
raw = load_dataset("AI-MO/NuminaMath-CoT", split="train")
print(f"Total rows: {len(raw)}")

# Filter: must have \boxed in solution, reasonable lengths
def is_valid_numinamath(ex):
    sol = ex.get("solution", "")
    prob = ex.get("problem", "")
    return (
        r"\boxed" in sol
        and 20 < len(prob) < 2000
        and 50 < len(sol) < 6000
    )

filtered = raw.filter(is_valid_numinamath, num_proc=4)
print(f"After filtering: {len(filtered)}")

# Random sample
n = min(NUMINAMATH_N, len(filtered))
indices = random.sample(range(len(filtered)), n)
sample = filtered.select(indices)
print(f"Sampled: {len(sample)}")

# Format
numinamath_texts = []
skipped = 0
for ex in sample:
    reasoning, boxed = extract_boxed_and_reasoning(ex["solution"])
    if not boxed:
        skipped += 1
        continue
    msgs = make_messages(SYSTEM_PROMPT_MATH, ex["problem"], reasoning, boxed)
    numinamath_texts.append(apply_chat_template(msgs))

print(f"NuminaMath formatted: {len(numinamath_texts)}  (skipped {skipped} with no \\boxed)")

In [ ]:
# Cell 7 — Load public correct responses (exp_004)
# Uploads files one at a time since they come from different local directories.

import json
from google.colab import files

print("Upload 1/3: data/public.jsonl")
files.upload()

print("Upload 2/3: experiments/exp_004_fewshot_prompts/public_responses.jsonl")
files.upload()

print("Upload 3/3: experiments/exp_004_fewshot_prompts/results.jsonl")
files.upload()

PUBLIC_JSONL_PATH       = "/content/public.jsonl"
PUBLIC_RESPONSES_PATH   = "/content/public_responses.jsonl"
RESULTS_PATH            = "/content/results.jsonl"

# Load ground-truth questions keyed by id
questions = {}
with open(PUBLIC_JSONL_PATH) as f:
    for line in f:
        ex = json.loads(line)
        questions[ex["id"]] = ex
print(f"Questions loaded: {len(questions)}")

# Load correct IDs from scored results
correct_ids = set()
with open(RESULTS_PATH) as f:
    for line in f:
        r = json.loads(line)
        if r.get("correct"):
            correct_ids.add(r["id"])
print(f"Correct IDs: {len(correct_ids)}")

# options is a list — map index to letter A, B, C...
LETTERS = "ABCDEFGHIJ"

# Format correct responses
public_texts = []
skipped = 0
with open(PUBLIC_RESPONSES_PATH) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] not in correct_ids:
            continue
        q = questions[r["id"]]
        system = SYSTEM_PROMPT_MCQ if r.get("is_mcq") else SYSTEM_PROMPT_MATH

        question_text = q["question"]
        if r.get("is_mcq") and "options" in q:
            opts = "\n".join(f"{LETTERS[i]}. {v}" for i, v in enumerate(q["options"]))
            question_text = f"{question_text}\n\nOptions:\n{opts}"

        reasoning, boxed = extract_boxed_and_reasoning(r["response"])
        if not boxed:
            skipped += 1
            continue
        msgs = make_messages(system, question_text, reasoning, boxed)
        public_texts.append(apply_chat_template(msgs))

print(f"Public correct formatted: {len(public_texts)}  (skipped {skipped})")

In [ ]:
# Cell 8 — Combine + build HuggingFace Dataset
from datasets import Dataset

all_texts = numinamath_texts + public_texts
random.shuffle(all_texts)
print(f"Total training examples: {len(all_texts)}")
print(f"  NuminaMath: {len(numinamath_texts)}")
print(f"  Public correct: {len(public_texts)}")

# Sanity check token lengths
lengths = [len(tokenizer.encode(t)) for t in all_texts[:500]]
import statistics
print(f"Token length (sample 500) — mean: {statistics.mean(lengths):.0f}, "
      f"p95: {sorted(lengths)[int(0.95*len(lengths))]}, max: {max(lengths)}")

# Filter examples that exceed MAX_SEQ_LENGTH
all_texts_filtered = [t for t in all_texts if len(tokenizer.encode(t)) <= MAX_SEQ_LENGTH]
print(f"After length filter: {len(all_texts_filtered)} "
      f"({len(all_texts)-len(all_texts_filtered)} dropped)")

train_dataset = Dataset.from_dict({"text": all_texts_filtered})
print(train_dataset)

In [ ]:
# Cell 9 — Train
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/content/exp008_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=50,
    save_strategy="epoch",
    seed=RANDOM_SEED,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,   # packs multiple short examples per 2048-token window — ~3-4x faster
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
# Cell 10 — Merge adapter into base model + save as float16
#
# This dequantizes the 4-bit base and adds the LoRA deltas,
# producing a standard float16 model that vLLM can load directly.

MERGED_PATH = "/content/exp008_merged"

print("Merging adapter into base weights (float16)...")
model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged model saved to {MERGED_PATH}")
!du -sh {MERGED_PATH}

In [ ]:
# Cell 11 — Push merged model to HuggingFace (private repo)
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

# Create private repo if it doesn't exist
try:
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"Repo creation note: {e}")

print("Uploading merged model (this will take a few minutes for ~8GB)...")
api.upload_folder(
    folder_path=MERGED_PATH,
    repo_id=HF_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Done! Model at: https://huggingface.co/{HF_REPO_ID}")
print(f"Set HF_REPO_ID = '{HF_REPO_ID}' in your Kaggle inference notebook.")